In [11]:
# Library yg akan dipakai

import os
import ast
import pandas as pd
import stat
import shutil
import re
import time
import subprocess
import csv
from urllib.parse import urlparse

from datetime import datetime

import json
import nbformat

import networkx as nx

In [12]:
akun_file = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\asli\list_dataset_github(10).txt"
projects_folder = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_mahasiswa(10)"
os.makedirs(projects_folder, exist_ok=True)

# Pengumpulan Data

dilakukan cloning repositori mahasiswa, diambil file python(.py dan .ipynb) saja

In [13]:
# untuk membuat log clone
success_count = 0
failed_count = 0
skipped_count = 0

log_file = "log_clone.csv"

# buat file log clone jika belum ada
if not os.path.exists(log_file):
    with open(log_file, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "repo_name", "repo_url", "status", "message"])


def write_log(repo_name, repo_url, status, message=""):
    with open(log_file, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            repo_name,
            repo_url,
            status,
            message
        ])

In [14]:
# Fungsi untuk membaca file txt yang berisi daftar link repo github Mahasiswa
import re

def load_repo_list(file_path):
    repo_list = []

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File {file_path} tidak ditemukan")

    with open(file_path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            # Hapus nomor urut seperti "1. "
            line = re.sub(r'^\d+\.\s*', '', line)

            repo_list.append(line)

    return repo_list

In [15]:
# Fungsi untuk mengambil file .py dan .ipynb saja

def keep_only_python_files(repo_path):
    # 1. Hapus file non .py dan .ipynb
    for root, dirs, files in os.walk(repo_path):
        if ".git" in dirs:
            dirs.remove(".git")

        for file in files:
            if not (file.endswith(".py") or file.endswith(".ipynb")):
                try:
                    os.remove(os.path.join(root, file))
                except PermissionError:
                    pass  # abaikan file yang tidak bisa dihapus

    # 2. Hapus folder kosong
    for root, dirs, files in os.walk(repo_path, topdown=False):
        if ".git" in dirs:
            dirs.remove(".git")

        if not os.listdir(root):
            try:
                os.rmdir(root)
            except OSError:
                pass

In [16]:
# Fungsi untuk mengambil nama repo dari URL
def extract_repo_name(repo_url):
    """
    Mengambil nama repository dari URL GitHub
    dan menangani kasus /tree/main/
    """
    # Hilangkan bagian /tree/ jika ada
    if "/tree/" in repo_url:
        repo_url = repo_url.split("/tree/")[0]

    # Hilangkan .git jika ada
    repo_url = repo_url.replace(".git", "")

    parsed = urlparse(repo_url)
    repo_name = os.path.basename(parsed.path)

    return repo_name

In [17]:
# Handler untuk file read-only (Windows fix)
def remove_readonly(func, path, exc_info):
    os.chmod(path, stat.S_IWRITE)
    func(path)

In [18]:
# Fungsi clone dan filter yang aman
def clone_and_filter_repo(repo_url, target_dir, delay=2):
    global success_count, failed_count, skipped_count
    
    repo_name = extract_repo_name(repo_url)
    folder_name = re.sub(r'[<>:"/\\|?* ]+', '_', repo_name)
    target_path = os.path.join(target_dir, folder_name)

    if os.path.exists(target_path):
        print(f"[SKIP] {folder_name} sudah ada.")
        write_log(folder_name, repo_url, "SKIPPED", "Folder sudah ada")
        skipped_count += 1
        return

    print(f"[CLONE] {folder_name}")

    try:
        subprocess.run(
            [
                "git",
                "-c", "core.protectNTFS=false",
                "-c", "core.longpaths=true",
                "clone",
                repo_url,
                target_path
            ],
            check=True,
            timeout=300,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE
        )

        keep_only_python_files(target_path)

        write_log(folder_name, repo_url, "SUCCESS", "Clone berhasil")
        success_count += 1

        time.sleep(delay)

    except subprocess.TimeoutExpired as e:
        print(f"\n--- (FAILED) Timeout saat clone {repo_url} ---")
        print("Alasan: Proses cloning melebihi batas waktu 300 detik")

        error_msg = "Timeout"

        cleanup_folder(target_path)

        write_log(folder_name, repo_url, "FAILED", error_msg)
        failed_count += 1

    except subprocess.CalledProcessError as e:
        print(f"\n--- (FAILED) Gagal clone {repo_url} ---")

        error_detail = e.stderr.decode(errors="ignore")
        print("Alasan error dari Git:")
        print(error_detail)

        cleanup_folder(target_path)

        write_log(folder_name, repo_url, "FAILED", error_detail)
        failed_count += 1


def cleanup_folder(target_path):
    if os.path.exists(target_path):
        try:
            shutil.rmtree(target_path, onerror=remove_readonly)
            print("🧹 Folder berhasil dihapus karena clone gagal")
        except Exception as cleanup_error:
            print("⚠ Tidak bisa menghapus folder!")
            print("Alasan gagal menghapus folder:")
            print(str(cleanup_error))

In [19]:
repo_mahasiswa = load_repo_list(akun_file)

# print(f"Total repo mahasiswa: {len(repo_mahasiswa)}\n")

for url in repo_mahasiswa:
    clone_and_filter_repo(
        repo_url=url,
        target_dir=projects_folder,
        delay=2
    )

print("\n--- Proses cloning selesai ---")
print(f"Dataset tersimpan di folder: {projects_folder}")
print(f"Total repo diproses : {success_count + failed_count + skipped_count}")
print(f"Berhasil clone      : {success_count}")
print(f"Gagal clone         : {failed_count}")
print(f"Skipped             : {skipped_count}")

[CLONE] 2341720040_ML_2025
[CLONE] 2341720131_ML_2025
[CLONE] 2341720217_ML_202525

--- (FAILED) Gagal clone https://github.com/NathanaelGracedo/2341720217_ML_202525 ---
Alasan error dari Git:
Cloning into 'D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_mahasiswa(10)\2341720217_ML_202525'...
remote: Repository not found.
fatal: repository 'https://github.com/NathanaelGracedo/2341720217_ML_202525/' not found

[CLONE] 2341720153_ML_2025
[CLONE] 2241720092_ML_2025
[CLONE] 2341720187_ML_2025
[CLONE] 2341720041_ML_2025
[CLONE] 244107023010_ML_2025

--- (FAILED) Gagal clone https://github.com/fajrulsantoso/244107023010_ML_2025 ---
Alasan error dari Git:
Cloning into 'D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_mahasiswa(10)\244107023010_ML_2025'...
error: unable to create file JS08 /JS08: No such file or directory
fatal: unable to checkout working tree
You can inspect what was checked out with 'git status'
and retry with 'git restore --source=HEAD :/'


🧹 Folder berhasil dihapus karena clon

## Normalisasi stuktur folder dan nama

In [20]:
import shutil

def normalize_repo_folder_name(repo_folder, base_path):
    """
    Mengubah nama folder repo menjadi NIM saja.
    Jika tidak sesuai pola NIM → folder dihapus.
    """

    old_path = os.path.join(base_path, repo_folder)

    # Ambil NIM di awal nama folder
    match_nim = re.match(r"^(\d{8,})", repo_folder)

    if not match_nim:
        print(f"❌ Nama repo tidak sesuai pola NIM → {repo_folder} DIHAPUS")
        shutil.rmtree(old_path)
        return None

    nim = match_nim.group(1)
    new_path = os.path.join(base_path, nim)

    # Jika sudah sesuai
    if repo_folder == nim:
        return nim

    # Jika folder NIM sudah ada → hapus repo lama
    if os.path.exists(new_path):
        print(f"⚠ Folder {nim} sudah ada → {repo_folder} DIHAPUS")
        shutil.rmtree(old_path)
        return None

    os.rename(old_path, new_path)
    print(f"🔄 Repo: {repo_folder} → {nim}")

    return nim

In [21]:
# VALIDATOR STRUKTUR

def is_valid_module_structure(student_path):
    """
    Mengecek apakah folder mahasiswa memiliki pola modul valid:
    - JSxx
    - KUIS
    - PBL_Kelompokxx
    - KELOMPOK
    """

    for item in os.listdir(student_path):
        if not os.path.isdir(os.path.join(student_path, item)):
            continue

        name = item.upper()

        if re.match(r"JS\d+", name):
            return True

        if name == "KUIS":
            return True

        if re.match(r"PBL[_ ]?KELOMPOK\d+", name):
            return True

        if name == "KELOMPOK":
            return True

    return False

In [22]:
# NORMALISASI NAMA FOLDER MODUL

def normalize_module_folder_name(folder_name):

    name = folder_name.upper()

    # JS01 → M01
    match_js = re.match(r"JS(\d+)", name)
    if match_js:
        number = int(match_js.group(1))
        return f"M{number:02d}"

    # KUIS → kuis
    match_kuis = re.match(r"KUIS[_ ]?(\d+)?", name)
    if match_kuis:
        return "kuis"

    # PBL_Kelompok2 → PBL02
    match_pbl = re.match(r"PBL[_ ]?KELOMPOK(\d+)", name)
    if match_pbl:
        number = int(match_pbl.group(1))
        return f"PBL{number:02d}"

    # KELOMPOK → kelompok
    if name == "KELOMPOK":
        return "kelompok"

    return None

In [23]:
# KLASIFIKASI FILE TUGAS

def classify_and_extract_practicum(filename):
    name = filename.lower()

    # Kelompok
    if "kelompok" in name:
        match_kel = re.search(r"kelompok[_ ]?(\d+)", name)
        if match_kel:
            return "KELOMPOK", int(match_kel.group(1))
        return "KELOMPOK", None

    # Kuis
    if "kuis" in name:
        return "KUIS", None

    # Tugas Praktikum
    if "tp" in name or "tugas" in name or "project" in name:
        return "TP", None

    # Praktikum P1, P2, dst
    match_p = re.search(r"\bp(\d+)", name)
    if match_p:
        return "PRAKTIKUM", int(match_p.group(1))

    return "PRAKTIKUM", None

In [24]:
# NORMALISASI NAMA FILE

def normalize_task_files(module_path):

    task_files = [
        f for f in os.listdir(module_path)
        if f.endswith(".py") or f.endswith(".ipynb")
    ]

    fallback_praktikum = 1
    tp_counter = 1
    kuis_counter = 1
    kelompok_counter = 1

    for file in task_files:

        old_path = os.path.join(module_path, file)
        task_type, number = classify_and_extract_practicum(file)
        ext = os.path.splitext(file)[1]

        # KELOMPOK
        if task_type == "KELOMPOK":
            if number is not None:
                new_name = f"Kelompok{number:02d}{ext}"
            else:
                new_name = f"Kelompok{kelompok_counter:02d}{ext}"
                kelompok_counter += 1

        # KUIS
        elif task_type == "KUIS":
            if kuis_counter == 1:
                new_name = f"Kuis{ext}"
            else:
                new_name = f"Kuis{kuis_counter:02d}{ext}"
            kuis_counter += 1

        # TP
        elif task_type == "TP":
            if tp_counter == 1:
                new_name = f"TP{ext}"
            else:
                new_name = f"TP{tp_counter:02d}{ext}"
            tp_counter += 1

        # PRAKTIKUM
        else:
            if number is not None:
                new_name = f"P{number:02d}{ext}"
            else:
                new_name = f"P{fallback_praktikum:02d}{ext}"
                fallback_praktikum += 1

        new_path = os.path.join(module_path, new_name)

        if old_path != new_path and not os.path.exists(new_path):
            os.rename(old_path, new_path)

In [25]:
def normalize_student_directory(student_path):
    student_name = os.path.basename(student_path)

    print(f"\n🔎 Cek struktur mahasiswa: {student_name}")

    # VALIDASI STRUKTUR
    if not is_valid_module_structure(student_path):

        print(f"❌ STRUKTUR TIDAK SESUAI → {student_name} DIHAPUS")

        # Hapus folder mahasiswa
        try:
            shutil.rmtree(student_path, onerror=remove_readonly)
            print(f"   🧹 Folder {student_name} berhasil dihapus")
        except Exception as e:
            print(f"   ⚠ Gagal menghapus folder {student_name}")
            print(f"   Alasan: {str(e)}")

        return

    print(f"✅ Struktur valid → mulai normalisasi {student_name}")

    for item in os.listdir(student_path):

        old_path = os.path.join(student_path, item)

        if not os.path.isdir(old_path):
            continue

        new_name = normalize_module_folder_name(item)

        if new_name is None:
            continue

        new_path = os.path.join(student_path, new_name)

        if old_path != new_path:
            print(f"   🔄 Folder: {item} → {new_name}")
            os.rename(old_path, new_path)
        else:
            print(f"   ✔ Folder: {item} (tidak berubah)")

        print(f"      📄 Normalisasi file di folder {new_name}")
        normalize_task_files(new_path)

    print(f"✅ Selesai normalisasi mahasiswa: {student_name}")

In [26]:
# LOOPING NORMALISASI SEMUA MAHASISWA

for repo_folder in os.listdir(projects_folder):

    old_repo_path = os.path.join(projects_folder, repo_folder)

    if not os.path.isdir(old_repo_path):
        continue

    # STEP 1: Normalisasi nama repo diambil NIM
    nim = normalize_repo_folder_name(repo_folder, projects_folder)

    if nim is None:
        continue

    student_path = os.path.join(projects_folder, nim)

    # STEP 2: Normalisasi struktur modul
    normalize_student_directory(student_path)

print("\n--- Normalisasi repo & struktur selesai ---")

🔄 Repo: 2241720092_ML_2025 → 2241720092

🔎 Cek struktur mahasiswa: 2241720092
❌ STRUKTUR TIDAK SESUAI → 2241720092 DIHAPUS
   🧹 Folder 2241720092 berhasil dihapus
🔄 Repo: 2341720040_ML_2025 → 2341720040

🔎 Cek struktur mahasiswa: 2341720040
✅ Struktur valid → mulai normalisasi 2341720040
   🔄 Folder: JS01 → M01
      📄 Normalisasi file di folder M01
   🔄 Folder: JS02 → M02
      📄 Normalisasi file di folder M02
   🔄 Folder: JS03 → M03
      📄 Normalisasi file di folder M03
   🔄 Folder: JS05 → M05
      📄 Normalisasi file di folder M05
   🔄 Folder: JS06 → M06
      📄 Normalisasi file di folder M06
   🔄 Folder: JS07 → M07
      📄 Normalisasi file di folder M07
   🔄 Folder: JS08 → M08
      📄 Normalisasi file di folder M08
   🔄 Folder: JS09 → M09
      📄 Normalisasi file di folder M09
   🔄 Folder: JS11 → M11
      📄 Normalisasi file di folder M11
   🔄 Folder: JS13 → M13
      📄 Normalisasi file di folder M13
   🔄 Folder: JS14 → M14
      📄 Normalisasi file di folder M14
   🔄 Folder: JS15 

# Preprocessing

In [27]:
preprocessed_folder = projects_folder + "_preprocessed"
os.makedirs(preprocessed_folder, exist_ok=True)

In [28]:
def clean_python_code(code):
    """
    Menghapus komentar dan docstring
    tanpa merusak struktur kode.
    """

    try:
        tree = ast.parse(code)
    except:
        return None  # Skip jika syntax error

    cleaned_lines = []

    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.ClassDef, ast.Module)):
            if (
                node.body
                and isinstance(node.body[0], ast.Expr)
                and isinstance(node.body[0].value, ast.Str)
            ):
                node.body.pop(0)  # Remove docstring

    cleaned_code = ast.unparse(tree)

    # Hapus komentar inline
    cleaned_code = re.sub(r"#.*", "", cleaned_code)

    # Hapus spasi berlebih & baris kosong
    lines = cleaned_code.splitlines()
    lines = [line.rstrip() for line in lines if line.strip() != ""]

    return "\n".join(lines)

In [29]:
def clean_python_code(code):
    try:
        tree = ast.parse(code)
    except:
        return None

    # Hapus docstring
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.ClassDef, ast.Module)):
            if (
                node.body
                and isinstance(node.body[0], ast.Expr)
                and isinstance(node.body[0].value, ast.Str)
            ):
                node.body.pop(0)

    cleaned_code = ast.unparse(tree)

    # Hapus komentar
    cleaned_code = re.sub(r"#.*", "", cleaned_code)

    # Hapus baris kosong
    lines = cleaned_code.splitlines()
    lines = [line.rstrip() for line in lines if line.strip() != ""]

    return "\n".join(lines)

In [30]:
def extract_and_clean_notebook(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            notebook = json.load(f)
    except:
        return None

    cleaned_blocks = []

    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        cleaned = clean_python_code(source)

        if cleaned:
            cleaned_blocks.append(cleaned)

    return "\n".join(cleaned_blocks)

In [31]:
print("\n🧹 Mulai preprocessing ke folder baru...")

file_processed = 0
file_skipped = 0

for root, dirs, files in os.walk(projects_folder):

    # Buat path tujuan
    relative_path = os.path.relpath(root, projects_folder)
    target_root = os.path.join(preprocessed_folder, relative_path)

    os.makedirs(target_root, exist_ok=True)

    for file in files:

        source_path = os.path.join(root, file)

        try:
            # === FILE PY ===
            if file.endswith(".py"):

                with open(source_path, "r", encoding="utf-8") as f:
                    code = f.read()

                cleaned = clean_python_code(code)

                if cleaned:
                    target_path = os.path.join(target_root, file)
                    with open(target_path, "w", encoding="utf-8") as f:
                        f.write(cleaned)
                    file_processed += 1
                else:
                    file_skipped += 1

            # === FILE IPYNB ===
            elif file.endswith(".ipynb"):

                cleaned = extract_and_clean_notebook(source_path)

                if cleaned:
                    new_filename = file.replace(".ipynb", ".py")
                    target_path = os.path.join(target_root, new_filename)

                    with open(target_path, "w", encoding="utf-8") as f:
                        f.write(cleaned)

                    file_processed += 1
                else:
                    file_skipped += 1

        except Exception as e:
            print(f"⚠ Error preprocessing: {source_path}")
            print(str(e))
            file_skipped += 1


print("\n===== RINGKASAN PREPROCESSING =====")
print(f"File berhasil diproses : {file_processed}")
print(f"File di-skip           : {file_skipped}")
print("====================================")
print(f"Dataset hasil preprocessing tersimpan di: {preprocessed_folder}")


🧹 Mulai preprocessing ke folder baru...

===== RINGKASAN PREPROCESSING =====
File berhasil diproses : 309
File di-skip           : 6
Dataset hasil preprocessing tersimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_mahasiswa(10)_preprocessed


In [34]:
normalized_folder = preprocessed_folder + "_normalized"
os.makedirs(normalized_folder, exist_ok=True)

In [32]:
class IdentifierNormalizer(ast.NodeTransformer):
    def __init__(self):
        self.var_map = {}
        self.func_map = {}
        self.arg_map = {}

        self.var_counter = 1
        self.func_counter = 1
        self.arg_counter = 1

    # ==========================
    # Normalisasi Nama Fungsi
    # ==========================
    def visit_FunctionDef(self, node):

        # Rename function
        if node.name not in self.func_map:
            new_name = f"func{self.func_counter}"
            self.func_map[node.name] = new_name
            self.func_counter += 1

        node.name = self.func_map[node.name]

        # Rename arguments
        for arg in node.args.args:
            if arg.arg not in self.arg_map:
                new_arg = f"arg{self.arg_counter}"
                self.arg_map[arg.arg] = new_arg
                self.arg_counter += 1
            arg.arg = self.arg_map[arg.arg]

        self.generic_visit(node)
        return node

    # ==========================
    # Normalisasi Variabel
    # ==========================
    def visit_Name(self, node):

        if isinstance(node.ctx, ast.Store) or isinstance(node.ctx, ast.Load):

            if node.id not in self.var_map and node.id not in self.arg_map:
                new_var = f"var{self.var_counter}"
                self.var_map[node.id] = new_var
                self.var_counter += 1

            if node.id in self.var_map:
                node.id = self.var_map[node.id]

            elif node.id in self.arg_map:
                node.id = self.arg_map[node.id]

        return node

In [33]:
def normalize_identifiers(code):

    try:
        tree = ast.parse(code)
    except:
        return None

    normalizer = IdentifierNormalizer()
    tree = normalizer.visit(tree)
    ast.fix_missing_locations(tree)

    try:
        return ast.unparse(tree)
    except:
        return None

In [35]:
print("\n🔁 Mulai normalisasi identifier...")

file_processed = 0
file_skipped = 0

for root, dirs, files in os.walk(preprocessed_folder):

    relative_path = os.path.relpath(root, preprocessed_folder)
    target_root = os.path.join(normalized_folder, relative_path)

    os.makedirs(target_root, exist_ok=True)

    for file in files:

        if not file.endswith(".py"):
            continue

        source_path = os.path.join(root, file)
        target_path = os.path.join(target_root, file)

        try:
            with open(source_path, "r", encoding="utf-8") as f:
                code = f.read()

            normalized = normalize_identifiers(code)

            if normalized:
                with open(target_path, "w", encoding="utf-8") as f:
                    f.write(normalized)

                file_processed += 1
            else:
                file_skipped += 1

        except Exception as e:
            print(f"⚠ Error normalisasi: {source_path}")
            print(str(e))
            file_skipped += 1


print("\n===== RINGKASAN IDENTIFIER NORMALIZATION =====")
print(f"File berhasil dinormalisasi : {file_processed}")
print(f"File di-skip                : {file_skipped}")
print("===============================================")
print(f"Hasil disimpan di: {normalized_folder}")


🔁 Mulai normalisasi identifier...

===== RINGKASAN IDENTIFIER NORMALIZATION =====
File berhasil dinormalisasi : 300
File di-skip                : 9
Hasil disimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\kode_mahasiswa(10)_preprocessed_normalized


In [36]:
def validate_python_file(file_path):

    # Cek ukuran file
    if os.path.getsize(file_path) == 0:
        return False, "File kosong"

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            code = f.read()

        if code.strip() == "":
            return False, "File kosong (blank)"

        ast.parse(code)

        return True, "Valid"

    except SyntaxError as e:
        return False, f"SyntaxError: {str(e)}"

    except Exception as e:
        return False, f"Error: {str(e)}"

In [37]:
print("\n🔎 Mulai validasi kode...")

valid_count = 0
invalid_count = 0
empty_count = 0

for root, dirs, files in os.walk(normalized_folder):

    for file in files:

        if not file.endswith(".py"):
            continue

        file_path = os.path.join(root, file)

        is_valid, reason = validate_python_file(file_path)

        if is_valid:
            valid_count += 1
        else:

            if "kosong" in reason.lower():
                empty_count += 1
            else:
                invalid_count += 1


print("\n===== RINGKASAN VALIDASI =====")
print(f"File valid           : {valid_count}")
print(f"File kosong          : {empty_count}")
print(f"File syntax error    : {invalid_count}")
print("================================")


🔎 Mulai validasi kode...

===== RINGKASAN VALIDASI =====
File valid           : 300
File kosong          : 0
File syntax error    : 0
